In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'orjson', 'polars', 'pyarrow'])
import os, gc, gzip, orjson, random, csv
import pandas as pd
import numpy as np
import polars as pl
import pyarrow.parquet as pq

In [ ]:
K_CORE = 5
TRAIN_RATIO = 0.8
RAW_REVIEW_PATH = '/kaggle/input/b22dckh072/file01/Clothing_Shoes_and_Jewelry.jsonl.gz'
RAW_META_PATH   = '/kaggle/input/b22dckh072/file01/meta_Clothing_Shoes_and_Jewelry.jsonl.gz'

WORKING_DIR = '/kaggle/working/processed'
os.makedirs(WORKING_DIR, exist_ok=True)

INTERIM_TRAIN_CSV = '/kaggle/working/interim_train.csv'
INTERIM_TEST_CSV  = '/kaggle/working/interim_test.csv'
TRAIN_PARQUET     = os.path.join(WORKING_DIR, 'train_interactions.parquet')
TEST_PARQUET      = os.path.join(WORKING_DIR, 'test_interactions.parquet')
META_PARQUET      = os.path.join(WORKING_DIR, 'filtered_metadata.parquet')

Paths configured.


In [ ]:
def fast_json_to_csv_split(input_path, train_path, test_path, ratio=0.8):
    import orjson
    print("Đang chuyển đổi và phân tách Train/Test theo luồng...")
    
    f_train = open(train_path, 'w', newline='')
    f_test = open(test_path, 'w', newline='')
    writer_train = csv.writer(f_train)
    writer_test = csv.writer(f_test)
    
    header = ['user_id', 'parent_asin', 'rating', 'timestamp']
    writer_train.writerow(header)
    writer_test.writerow(header)

    with gzip.open(input_path, 'rb') as f:
        for line in f:
            data = orjson.loads(line)
            row = [data['user_id'], data['parent_asin'], data['rating'], data['timestamp']]
            
            if np.random.random() < ratio:
                writer_train.writerow(row)
            else:
                writer_test.writerow(row)
                
    f_train.close()
    f_test.close()
    print("-> Xong bước tách dữ liệu tạm thời.")

fast_json_to_csv_split(RAW_REVIEW_PATH, INTERIM_TRAIN_CSV, INTERIM_TEST_CSV, TRAIN_RATIO)

In [ ]:
def apply_k_core(train_path, k=5):
    print(f"Đang áp dụng K-Core={k} bằng Polars (Tối ưu RAM)...")
    
    # CHỈ ĐỌC 2 CỘT CẦN THIẾT (Bỏ cột rating và timestamp rất nặng)
    lf = pl.scan_csv(train_path).select(['user_id', 'parent_asin'])
    
    # Dùng lazy evaluation tối đa trước khi collect
    curr_train = lf.collect()
    
    for i in range(3): 
        print(f"   Bắt đầu vòng lặp K-Core thứ {i+1}...")
        
        # 1. Đếm và lọc User
        user_counts = curr_train.group_by('user_id').len()
        valid_users = user_counts.filter(pl.col('len') >= k).select('user_id')
        
        # Ghi đè curr_train ngay lập tức để tiết kiệm RAM
        curr_train = curr_train.join(valid_users, on='user_id', how='inner')
        del user_counts, valid_users
        gc.collect()
        
        # 2. Đếm và lọc Item
        item_counts = curr_train.group_by('parent_asin').len()
        valid_items = item_counts.filter(pl.col('len') >= k).select('parent_asin')
        
        curr_train = curr_train.join(valid_items, on='parent_asin', how='inner')
        del item_counts, valid_items
        gc.collect()
        
        print(f"   -> Cuối vòng {i+1}: Còn lại {curr_train.height:,} tương tác.")
        
    # Chỉ trả về ID duy nhất để làm bảng Mapping
    return curr_train.unique()

valid_ids = apply_k_core(INTERIM_TRAIN_CSV, K_CORE)
valid_user_set = set(valid_ids['user_id'].to_list())
valid_item_set = set(valid_ids['parent_asin'].to_list())

print(f"Số lượng User hợp lệ: {len(valid_user_set):,}")
print(f"Số lượng Item hợp lệ: {len(valid_item_set):,}")

del valid_ids
gc.collect()

In [ ]:
print("Đang khởi tạo bảng Mapping và xuất file Parquet...")

# Tạo bảng map
u_map = pl.DataFrame({'user_id': list(valid_user_set)}).with_row_index('mapped_user_id', offset=1)
i_map = pl.DataFrame({'parent_asin': list(valid_item_set)}).with_row_index('mapped_item_id', offset=1)

# Ép kiểu dữ liệu nhỏ để tiết kiệm RAM cho các File sau (Int32)
u_map = u_map.with_columns(pl.col('mapped_user_id').cast(pl.Int32))
i_map = i_map.with_columns(pl.col('mapped_item_id').cast(pl.Int32))

# Map và lưu Train
(pl.scan_csv(INTERIM_TRAIN_CSV)
 .join(u_map.lazy(), on='user_id', how='inner')
 .join(i_map.lazy(), on='parent_asin', how='inner')
 .select(['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp'])
 .sink_parquet(TRAIN_PARQUET))

# Map và lưu Test (Chỉ giữ lại những User/Item đã có trong tập Train để tránh Cold-start)
(pl.scan_csv(INTERIM_TEST_CSV)
 .join(u_map.lazy(), on='user_id', how='inner')
 .join(i_map.lazy(), on='parent_asin', how='inner')
 .select(['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp'])
 .sink_parquet(TEST_PARQUET))

print(f"-> Đã lưu: {TRAIN_PARQUET} và {TEST_PARQUET}")

# Dọn dẹp file CSV tạm
os.remove(INTERIM_TRAIN_CSV)
os.remove(INTERIM_TEST_CSV)

In [6]:
def process_meta_chunked(meta_path, valid_item_ids, output_path):
    """
    Lọc Meta Data bằng Streaming & Batching.
    Áp dụng kỹ thuật KHÓA BỘ ĐỆM GZIP giống Bước 1 để chống tràn RAM tuyệt đối.
    """
    try:
        import ujson as json_lib
    except ImportError:
        import orjson as json_lib

    valid_items_set = set(valid_item_ids)
    
    print("Đang lọc Meta Data bằng Streaming (Chống tràn RAM)...")
    
    # KẾT HỢP XẢ BỘ ĐỆM: Không truyền trực tiếp f_out vào csv.writer trong block 'with'
    f_out = open(output_path, 'w', newline='', encoding='utf-8')
    writer = csv.writer(f_out)
    writer.writerow(['parent_asin', 'price', 'main_category', 'store', 'average_rating', 'rating_number', 'title', 'images'])
    
    batch = []
    
    # KHÓA BỘ ĐỆM GZIP: Dùng GzipFile đọc byte thuần túy thay vì 'rt'
    with gzip.GzipFile(meta_path, 'r') as f_in:
        for line in f_in:
            try:
                # Đọc byte và decode để hệ thống xả rác liên tục
                data = json_lib.loads(line.decode('utf-8').strip())
            except:
                continue
                
            asin = data.get('parent_asin')
            if asin in valid_items_set:
                batch.append([
                    asin,
                    data.get('price', ''),
                    data.get('main_category', ''),
                    data.get('store', ''),
                    data.get('average_rating', ''),
                    data.get('rating_number', ''),
                    data.get('title', ''),
                    str(data.get('images', '')) # Ép kiểu mảng ảnh thành chuỗi
                ])
                
            # Ép xả bộ nhớ khi đạt 200.000 dòng
            if len(batch) >= 200000:
                writer.writerows(batch)
                batch.clear()
                
                f_out.flush()
                os.fsync(f_out.fileno())
                gc.collect()
                
        # Ghi nốt phần dữ liệu còn lẻ ở cuối
        if batch:
            writer.writerows(batch)
            f_out.flush()
            os.fsync(f_out.fileno())
            batch.clear()
            gc.collect()
            
    f_out.close()
    print("-> Lọc Meta hoàn tất an toàn!")

In [ ]:
def process_meta_to_parquet(raw_meta_path, valid_items, out_path):
    import orjson
    print("Đang lọc Metadata và nén sang Parquet...")
    temp_csv = out_path + ".tmp"
    
    with open(temp_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['parent_asin', 'price', 'average_rating', 'rating_number', 'store'])
        
        with gzip.open(raw_meta_path, 'rb') as f_in:
            for line in f_in:
                data = orjson.loads(line)
                asin = data.get('parent_asin')
                if asin in valid_items:
                    writer.writerow([
                        asin, 
                        data.get('price', 0), 
                        data.get('average_rating', 0),
                        data.get('rating_number', 0),
                        data.get('store', 'Unknown')
                    ])
                    
    # Dùng Polars nén CSV tạm sang Parquet cực nhanh
    pl.read_csv(temp_csv).join(i_map, on='parent_asin', how='inner').sink_parquet(out_path)
    os.remove(temp_csv)

process_meta_to_parquet(RAW_META_PATH, valid_item_set, META_PARQUET)
print("HOÀN TẤT TOÀN BỘ FILE 1!")